In [1]:
import sys
import numpy as np

# sys.path.append("../../../src/")
from Rain.Rain import Rain
# sys.path.pop()

from keras.models import Sequential
from keras.layers import Dense, Activation, Dropout
import tensorflow as tf

2023-07-05 02:35:09.839544: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2023-07-05 02:35:11.222088: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
import os

def clean():
    folder_paths = ["logs", "../../../data/coord/", "../../../data/divider/", "../../../data/worker/"]  # Replace with the folder path you want to delete files from
    file_extensions = [".npy", ".pkl", ".log"]  # Replace with the file extension you want to delete
    for folder_path in folder_paths:
        if os.path.exists(folder_path):
            for filename in os.listdir(folder_path):
                for file_extension in file_extensions:
                    if filename.endswith(file_extension):
                        file_path = os.path.join(folder_path, filename)
                        os.remove(file_path)
clean()

In [3]:
config = {
  "mode": {
      "type": "local",
      "params": {
        "ips": ['127.0.0.1', '127.0.0.1', '127.0.0.1'],
        "ports": [50151, 50152, 50153]
      }
      
    },
  "temp_data_path": "../../../",
  "partitions": 3,
  "num_of_workers": 3,
  "iterations": 3,
  "learning_type": "DL",
  "DL": {
    "lib": {
      "type": "tensorflow",
      "params": {
        "loss": tf.keras.losses.CategoricalCrossentropy(),
        "optimizer": tf.keras.optimizers.Adam(learning_rate=0.001),

      }
    },
    "lr": 0.001,
    "epochs": 1,
    "batch_size": 128,
  }
}

In [4]:
def get_train_data():
    return np.load("../../../data/MNIST/train_data.npy"), np.load(
        "../../../data/MNIST/train_labels.npy"
    )


def get_test_data():
    return np.load("../../../data/MNIST/test_data.npy"), np.load(
        "../../../data/MNIST/test_labels.npy"
    )



In [5]:
def create_model():
    # network parameters
    hidden_units = 256
    dropout = 0.45
    input_size = 784
    num_labels = 10
    # model is a 3-layer MLP with ReLU and dropout after each layer
    model = Sequential()
    model.add(Dense(hidden_units, input_dim=input_size))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(hidden_units))
    model.add(Activation("relu"))
    model.add(Dropout(dropout))
    model.add(Dense(num_labels))
    model.add(Activation("softmax"))
    return model

In [6]:
X_train, y_train = get_train_data()


In [7]:
model = create_model()
rain = Rain(config, model)

2023-07-05 02:35:12,689 [DEBUG] [Rain] Rain is initialized
2023-07-05 02:35:12,691 [DEBUG] [Provisioner] Creating coordinator
2023-07-05 02:35:12,693 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/coord/
2023-07-05 02:35:12,695 [DEBUG] [Coordinator] Coordinator is initialized
2023-07-05 02:35:12,696 [DEBUG] [LocalProvisioner] Provisioner is initialized
2023-07-05 02:35:12,696 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-05 02:35:12,698 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/
2023-07-05 02:35:12,699 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/divider/


In [8]:
model = rain.train(X_train, y_train, strategy='async')

2023-07-05 02:35:12,704 [DEBUG] [Rain] Creating workers
2023-07-05 02:35:12,710 [INFO] [Provisioner] provisioner is serving
2023-07-05 02:35:12,711 [DEBUG] [Provisioner] Starting coordinator
2023-07-05 02:35:12,713 [INFO] [Coordinator] coordinator is serving
2023-07-05 02:35:12,714 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-05 02:35:12,718 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-05 02:35:12,721 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-05 02:35:12,725 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-05 02:35:12,729 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker/
2023-07-05 02:35:12,733 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-05 02:35:12,735 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker/
2023-07-05 02:35:12,737 [INFO] [Worker_50

157/157 [==============================] - 3s 13ms/step - loss: 0.7057 - accuracy: 0.7776


2023-07-05 02:35:28,482 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on worker{worker_id}
2023-07-05 02:35:28,487 [DEBUG] [DividerAmbassador] divider begins downloading ../../..//RainData/divider/2_2_trained.pkl from worker2


sending data to divider


2023-07-05 02:35:28,637 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
2023-07-05 02:35:28,647 [DEBUG] [DeepLearning] Asynchronous update is done by worker 2
2023-07-05 02:35:28,672 [DEBUG] [DeepLearning] Iteration 1/3 complete for worker 2.
2023-07-05 02:35:28,673 [DEBUG] [DeepLearning] Starting iteration 2/3
2023-07-05 02:35:28,673 [DEBUG] [DividerAmbassador] 127.0.0.1:50152
2023-07-05 02:35:28,674 [DEBUG] [DividerAmbassador] divider begins will not send data in iteration2 to worker2
2023-07-05 02:35:28,675 [DEBUG] [DividerAmbassador] Sending ../../..//RainData/divider/2.pkl to worker2
2023-07-05 02:35:33,184 [DEBUG] [DividerAmbassador] divider received: File downloaded successfully after sending the model to worker 2
2023-07-05 02:35:33,185 [DEBUG] [DividerAmbassador] divider begins executing iteration2 for worker2
2023-07-05 02:35:36,348 [DEBUG] [DividerAmbassador] divider received: Executed! after executing the model on 

Error in loading the data: Error in loading the data:  Layer 'sequential' expected 0 variables, but received 1 variables during loading. Expected: []
 Layer 'sequential' expected 0 variables, but received 1 variables during loading. Expected: []
Error in receiving the data:  'NoneType' object is not subscriptable
Error in receiving the data:  'NoneType' object is not subscriptable


2023-07-05 02:35:39,306 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
2023-07-05 02:35:39,312 [DEBUG] [DeepLearning] Error in calculating the new weights: list index out of range
2023-07-05 02:35:39,327 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-05 02:35:39,335 [DEBUG] [DeepLearning] Error in calculating the new weights: list index out of range
2023-07-05 02:35:39,336 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-05 02:35:39,342 [DEBUG] [DeepLearning] Iteration 2/3 complete for worker 2.
2023-07-05 02:35:39,343 [DEBUG] [DeepLearning] Error in calculating the new weights: list index out of range
2023-07-05 02:35:39,344 [DEBUG] [DeepLearning] Starting iteration 3/3
2023-07-05 02:35:39,346 [DEBUG] [DividerAmbassador] 127.0.0.1:50152
2023-07-05 02:35:39,349 [DEBUG] [DividerAmbassador] div

Error in loading the data:  Layer 'sequential' expected 0 variables, but received 1 variables during loading. Expected: []
Error in receiving the data:  'NoneType' object is not subscriptable
Error in loading the data:  Layer 'sequential' expected 0 variables, but received 1 variables during loading. Expected: []
Error in receiving the data:  'NoneType' object is not subscriptable


2023-07-05 02:35:40,266 [DEBUG] [DeepLearning] Error in calculating the new weights: list index out of range
2023-07-05 02:35:40,303 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 2.
2023-07-05 02:35:40,314 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-05 02:35:40,318 [DEBUG] [DeepLearning] Error in calculating the new weights: list index out of range
2023-07-05 02:35:40,321 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_1_trained.pkl from worker1 successfully
2023-07-05 02:35:40,328 [DEBUG] [DeepLearning] Error in calculating the new weights: list index out of range
2023-07-05 02:35:40,349 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 3.
2023-07-05 02:35:40,357 [DEBUG] [DeepLearning] Iteration 3/3 complete for worker 1.
2023-07-05 02:35:40,360 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
2023-07-05 02:35:40,361 [DEBUG] [Divider] Divider stopped serving
2023

In [9]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 1s 7ms/step - loss: 0.4613 - accuracy: 0.9188

Test accuracy: 91.9%


In [10]:
model = rain.train(X_train, y_train, strategy='sync')

2023-07-05 02:35:41,181 [DEBUG] [Rain] Creating workers
2023-07-05 02:35:41,188 [INFO] [Provisioner] provisioner is serving
2023-07-05 02:35:41,189 [DEBUG] [Provisioner] Starting coordinator
2023-07-05 02:35:41,191 [INFO] [Coordinator] coordinator is serving
2023-07-05 02:35:41,192 [DEBUG] [Coordinator] sending the num of workers to the provisioner
2023-07-05 02:35:41,195 [DEBUG] [Provisioner] Received 'NumOfWorkers: 3
' from the coordinator to define the number of workers
2023-07-05 02:35:41,196 [DEBUG] [Coordinator] sent Success receiving the number of workers to the provisioner
2023-07-05 02:35:41,197 [DEBUG] [LocalProvisioner] Creating 3 workers
2023-07-05 02:35:41,198 [DEBUG] [TemporaryFilesManager] Created temporary directory ../../..//RainData/worker/
2023-07-05 02:35:41,199 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-05 02:35:41,199 [INFO] [Worker_50151] Worker is running on port: 50151
2023-07-05 02:35:41,201 [DEBUG] [TemporaryFilesManager] Created temporary

Error in loading the data:  Layer 'sequential' expected 0 variables, but received 1 variables during loading. Expected: []
Error in receiving the data:  'NoneType' object is not subscriptable
Error in loading the data:  Layer 'sequential' expected 0 variables, but received 1 variables during loading. Expected: []
Error in receiving the data:  'NoneType' object is not subscriptable


2023-07-05 02:35:52,885 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_2_trained.pkl from worker2 successfully
2023-07-05 02:35:52,893 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_2_trained.pkl from worker3 successfully
2023-07-05 02:35:52,912 [DEBUG] [DeepLearning] Error in calculating the new weights: list index out of range
2023-07-05 02:35:52,912 [DEBUG] [DeepLearning] Iteration 2/3 complete.
2023-07-05 02:35:52,913 [DEBUG] [DeepLearning] Starting iteration 3/3
2023-07-05 02:35:52,936 [DEBUG] [DividerAmbassador] 127.0.0.1:50151
2023-07-05 02:35:52,937 [DEBUG] [DividerAmbassador] 127.0.0.1:50152
2023-07-05 02:35:52,937 [DEBUG] [DividerAmbassador] 127.0.0.1:50153
2023-07-05 02:35:52,938 [DEBUG] [DividerAmbassador] divider begins will not send data in iteration3 to worker1
2023-07-05 02:35:52,939 [DEBUG] [DividerAmbassador] divider begins will not send data in iteration3 to worker2
2023-07-05 02:35:52,940 [DEBUG] [DividerAmbassador] divider 

Error in loading the data:  Ran out of input
Error in receiving the data:  'NoneType' object is not subscriptable
Error in loading the data:  Layer 'sequential' expected 0 variables, but received 1 variables during loading. Expected: []
Error in loading the data:  Layer 'sequential' expected 0 variables, but received 1 variables during loading. Expected: []
Error in receiving the data:  'NoneType' object is not subscriptable
Error in receiving the data:  'NoneType' object is not subscriptable


2023-07-05 02:35:53,551 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/1_3_trained.pkl from worker1 successfully
2023-07-05 02:35:53,587 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/2_3_trained.pkl from worker2 successfully
2023-07-05 02:35:53,589 [DEBUG] [DividerAmbassador] Downloaded ../../..//RainData/divider/3_3_trained.pkl from worker3 successfully
2023-07-05 02:35:53,606 [DEBUG] [DeepLearning] Error in calculating the new weights: list index out of range
2023-07-05 02:35:53,607 [DEBUG] [DeepLearning] Iteration 3/3 complete.
2023-07-05 02:35:53,608 [DEBUG] [DividerAmbassador] divider ambassador stopped serving
2023-07-05 02:35:53,609 [DEBUG] [Divider] Divider stopped serving
2023-07-05 02:35:53,609 [INFO] [Worker_50151] Worker stopped serving on port: 50151
2023-07-05 02:35:53,609 [INFO] [Worker_50151] Worker stopped serving on port: 50151
2023-07-05 02:35:53,610 [INFO] [Worker_50152] Worker stopped serving on port: 50152
2023-07-05 02:35:53

In [11]:
X_test, y_test = get_test_data()
loss, acc = model.evaluate(X_test, y_test, batch_size=config["DL"]["batch_size"])
print("\nTest accuracy: %.1f%%" % (100.0 * acc))

79/79 [==============================] - 0s 6ms/step - loss: 0.4613 - accuracy: 0.9188

Test accuracy: 91.9%


2023-07-05 02:36:22,976 [DEBUG] [Coordinator] coordinator is sending workers info to divider
